# Inleiding

HBHavens exporteert resultaten naar databases (HRD, HLCD, config) voor gebruik in Riskeer. Per locatie geeft de gebruiker een verwijzing naar een HRD database op waar de resultaten in worden geschreven. 
Het HLCD bestand bevat een koppeling tussen normtrajecten ('Tracks') en HRD databases. Deze koppeling wordt bij het exporteren naar een nieuwe database niet aangepast aan de nieuwe database naam die is opgegeven in HBHavens. Hierdoor kan Riskeer de nieuwe database niet vinden en rekent Riskeer geen belastingen uit. 

De code van HBHavens is aangepast zodat voor iedere unieke opgegeven SQLite export database (in tabblad Exporteren), de koppeling aan de HLCD wordt toegevoegd. Dit gebeurt door het TrackID van opgegeven database op te zoeken, en voor deze TrackID in de HLCD de naam van de database te wijzigen naar de naam zoals opgegeven in HBHavens. De oorspronkelijke database naam wordt dus overschreven (omdat de TrackID uniek moet zijn).

Code wijzigingen:
- functie toegevoegd om de aanpassing in de SQLite db te maken: io.database.set_track_hrdfilename
- Regels toegevoegd aan model.export.export_output_to_database() om bovenstaande aan te voegen voor alle databases, als de optie "export_HLCD_and_config" aangevinkt is.



# Imports

In [1]:
import sys
import os
import shutil

owd = os.getcwd()
testdir = os.path.dirname(owd)
sys.path.append(testdir)
trunkdir = os.path.dirname(testdir)
sys.path.append(trunkdir)

from hbhavens.core.models import MainModel
from hbhavens.core.geometry import extend_point_to_linestring

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import warnings
warnings.simplefilter("ignore")

# Schematisatie

In [2]:
#==============================================================================
# import harbor geometry
#==============================================================================
tempdir = os.path.join(os.path.dirname(trunkdir),"temp")
if not os.path.exists(tempdir):
    os.mkdir(tempdir)

case = 'vlissingen'
if os.path.isdir(os.path.join(tempdir, case)):
    shutil.rmtree(os.path.join(tempdir, case), ignore_errors=True)

datadir = os.path.join(trunkdir, 'tests', 'data', case)

harbor = MainModel()
harbor.project.settings['project']['name'] = case
harbor.project.settings['project']['user']['email'] = 'benit@hkv.nl'
harbor.project.settings['project']['user']['name'] = 'Matthijs Bénit'

harbor.project.save_as(os.path.join(tempdir, case) + '.json')

# Flooddefence
trajecten = ['29-2','29-3']
for trajectid in trajecten:
    harbor.schematisation.add_flooddefence(trajectid)

# Harbourarea
harborarealoc = os.path.join(datadir, 'shapes', 'haventerrein.shp')
harbor.schematisation.add_harborarea(harborarealoc)

if True:
    # 2 Breakwaters
    breakwaterloc = os.path.join(datadir, 'shapes', 'havendammen_test2.shp')
    harbor.schematisation.add_breakwater(breakwaterloc)
    harbor.schematisation.set_breakwater_heights([2., 1.])
    harbor.schematisation.set_breakwater_alphas([2.6, 2.4])
    harbor.schematisation.set_breakwater_betas([0.2, 0.15])
else:
    # 1 Breakwater
    breakwaterloc = os.path.join(datadir, 'shapes', 'havendammen.shp')
    harbor.schematisation.add_breakwater(breakwaterloc)
    harbor.schematisation.set_breakwater_heights([2.])
    harbor.schematisation.set_breakwater_alphas([2.6])
    harbor.schematisation.set_breakwater_betas([0.2])
    harbor.schematisation.set_entrance_coordinate((31019., 385255.)) # eindpunt zuidelijke golfbreker

# Harbour boundary
harbor.schematisation.generate_harbor_bound()

# Output locations
# harbor.schematisation.generate_result_locations(distance=50, interval=100, interp_length=25)
harbor.schematisation.add_result_locations(os.path.join(datadir, 'shapes', 'uitvoerlocaties.shp'))

# Bed level
bedlevel = -99.0
harbor.schematisation.set_bedlevel(bedlevel)

# Add database
database = os.path.join(datadir, 'databases', 'WBI2017_Westerschelde_29-3_29-4_v01.sqlite')
harbor.hydraulic_loads.add_HRD(database)
supportlocation = 'WS_1_29-3_dk_00024'
harbor.schematisation.set_selected_support_location(supportlocation)

harbor.save_tables()
harbor.project.save()

# Run simple

In [3]:

#==============================================================================
# run simple calculation
#==============================================================================
harbor.project.settings['calculation_method']['method'] = 'simple'
harbor.simple_calculation.initialize()
harbor.simple_calculation.diffraction.run()
harbor.simple_calculation.transmission.run()
harbor.simple_calculation.wavegrowth.run()
harbor.simple_calculation.wavebreaking.run()
harbor.simple_calculation.combinedresults.run(processes=['Diffractie', 'Transmissie', 'Lokale golfgroei'])

harbor.project.settings['simple']['finished'] = True
harbor.simple_calculation.save_results()


In [4]:
# #==============================================================================
# # Uncertainties
# #==============================================================================
harbor.modeluncertainties.add_result_locations()
harbor.modeluncertainties.load_modeluncertainties()

uncertainties = harbor.modeluncertainties.supportloc_unc.unstack()
uncertainties.index = [' '.join(vals) for vals in uncertainties.index]
harbor.modeluncertainties.table.loc[:, uncertainties.index] = uncertainties.values
harbor.modeluncertainties.save_tables()


# Export

In [5]:
# #==============================================================================
# # Export
# #==============================================================================
harbor.project.settings["export"]["HRD"] = os.path.join(tempdir, case, 'WBI2017_Westerschelde_29-3_29-4_v01.sqlite')
harbor.project.settings["export"]["export_HLCD_and_config"] = True
harbor.project.settings['export']['HLCD'] = os.path.join(tempdir, case, 'hlcd.sqlite')

shutil.copy2(os.path.join(datadir, 'databases', 'WBI2017_Westerschelde_29-3_29-4_v01.sqlite'), os.path.join(tempdir, case))
shutil.copy2(os.path.join(datadir, 'databases', 'WBI2017_Westerschelde_29-3_29-4_v01.config.sqlite'), os.path.join(tempdir, case))
shutil.copy2(os.path.join(trunkdir, 'tests', 'data', 'hlcd.sqlite'), os.path.join(tempdir, case))

harbor.export.add_result_locations()
harbor.export.export_dataframe['Exportnaam'] = case + '_' + harbor.export.export_dataframe['Naam'] 
harbor.export.export_output_to_database()
harbor.project.settings['export']['export_succeeded'] = True


'c:\\Users\\Benit\\HB Havens R489\\temp\\vlissingen\\WBI2017_Westerschelde_29-3_29-4_v01.sqlite'

'c:\\Users\\Benit\\HB Havens R489\\temp\\vlissingen\\WBI2017_Westerschelde_29-3_29-4_v01.config.sqlite'

'c:\\Users\\Benit\\HB Havens R489\\temp\\vlissingen\\hlcd.sqlite'

In [6]:
# Save results
harbor.project.save()
harbor.save_tables()